# NumPy Boolean Masking & Filtering

## Fokus Bab

Bab ini membahas teknik filtering data numerik tanpa loop eksplisit menggunakan boolean array sebagai index. Kemampuan ini merupakan salah satu keunggulan NumPy yang paling sering dipakai dalam praktik Data Science — mulai dari membersihkan data kotor, mendeteksi outlier, hingga melakukan transformasi kondisional pada dataset besar secara efisien.

## Tujuan Pembelajaran

* Membuat boolean array menggunakan operator perbandingan (>, <, >=, <=, ==, !=).
* Melakukan filtering data dengan boolean masking (arr[kondisi]) dan menjelaskan cara kerjanya secara internal.
* Menggabungkan lebih dari satu kondisi menggunakan operator &, |, ~ beserta aturan penulisannya yang benar.
* Melakukan penggantian nilai secara kondisional, baik melalui boolean assignment maupun np.where().
* Menjelaskan perbedaan mendasar antara hasil boolean masking (copy) dengan hasil slicing (view) yang telah dipelajari di bab 2.
* Menerapkan filtering untuk kasus nyata: deteksi outlier sederhana dan pembersihan data numerik.

## Goals

Mampu melakukan filtering dan conditional transformation pada data numerik menggunakan boolean masking — sekaligus memahami bahwa hasilnya berupa copy independen, bukan view seperti pada slicing (bab 2). Kemampuan ini menjadi fondasi untuk broadcasting dan vectorization pada bab 4 dan 5.

# Boolean Array — Operator Perbandingan

Fondasi dari boolean masking adalah boolean array, yaitu array yang berisi nilai True/False, dihasilkan dari penerapan operator perbandingan terhadap array numerik secara element-wise.

In [1]:
import numpy as np  

arr = np.array([3, -7, 12, -4, 9])

print(arr > 0)    # [True, False, True, False, True]
print(arr < 0)    # [False, True, False, True, False]
print(arr >= 9)   # [False, False, True, False, True]
print(arr == -7)  # [False, True, False, False, False]
print(arr != 0)   # [True, True, True, True, True])

[ True False  True False  True]
[False  True False  True False]
[False False  True False  True]
[False  True False False False]
[ True  True  True  True  True]


# Boolean Masking — Advanced Indexing

![Boolean array](../assets/figures/numpy_boolean_masking_pipeline.png)

Setelah boolean array terbentuk, langkah selanjutnya adalah menggunakannya sebagai index untuk memfilter array asli:

In [2]:
arr = np.array([3, -7, 12, -4, 9])
arr[arr > 0]   # [3, 12, 9]

array([ 3, 12,  9])

Cara kerja arr[arr > 0] dapat dipecah menjadi dua tahap konseptual:

* Tahap 1 — arr > 0 dievaluasi terlebih dahulu, menghasilkan boolean array [True, False, True, False, True].
* Tahap 2 — boolean array tersebut dipakai sebagai index (arr[boolean_array]), sehingga NumPy hanya mengambil elemen pada posisi yang bernilai True.

Diagram di atas menggambarkan alur dua tahap tersebut secara visual: array asli → boolean mask (hasil kondisi) → array hasil filtering yang hanya berisi elemen bernilai True.

# Deep Dive — Boolean Indexing Mengembalikan Copy, Bukan View

Ini adalah salah satu poin paling krusial yang sering terlewat oleh pemula, terutama setelah mempelajari konsep view pada bab 2. Berdasarkan dokumentasi resmi NumPy mengenai indexing:

* Boolean masking termasuk dalam kategori advanced indexing, berbeda dari basic slicing (arr[1:4]) yang dipelajari di bab 2.
* Advanced indexing selalu mengembalikan salinan (copy) data, berbeda dengan basic slicing yang selalu mengembalikan view dari array asli. 
* Konsekuensinya, jika Anda memodifikasi hasil dari arr[arr > 0], array aslinya tidak akan ikut berubah — berbeda total dengan perilaku slicing yang dibahas di bab 2.

In [4]:
arr = np.array([3, -7, 12, -4, 9])
positif = arr[arr > 0]   # copy, bukan view
positif[0] = 999

print(arr)      # [3, -7, 12, -4, 9]  → tidak berubah
print(positif)  # [999, 12, 9]

[ 3 -7 12 -4  9]
[999  12   9]


* Fakta teknis lain yang perlu diketahui: ketika dimensi boolean mask sama dengan dimensi array yang diindeks, hasilnya berupa array 1D yang berisi elemen-elemen bernilai True, diambil dengan urutan row-major (gaya C). 
* NumPy akan memunculkan IndexError apabila shape dari boolean mask tidak sesuai dengan dimensi array yang diindeks, terlepas dari nilai True/False di dalamnya. Artinya, shape boolean mask harus identik dengan shape array yang difilter.

Tabel Perbandingan Basic Slicing dan Boolean Masking

| Aspek                | Basic Slicing (`arr[1:4]`) | Boolean Masking (`arr[arr>0]`) |
| -------------------- | -------------------------- | ------------------------------ |
| **Kategori**         | Basic indexing             | Advanced indexing              |
| **Hasil**            | View (berbagi memori)      | Copy (independen)              |
| **Modifikasi hasil** | Memengaruhi array asli     | Tidak memengaruhi array asli   |
| **Bentuk hasil**     | Sesuai slice yang diminta  | 1D, hanya elemen `True`        |


# Multiple Conditions

Untuk menggabungkan lebih dari satu kondisi filtering, NumPy tidak menggunakan kata kunci Python biasa (and, or, not), melainkan operator bitwise:

| Operator | Makna              | Setara dengan           |
| -------- | ------------------ | ----------------------- |
| `&`      | AND *element-wise* | `and` pada Python biasa |
| `\|`     | OR *element-wise*  | `or` pada Python biasa  |
| `~`      | NOT *element-wise* | `not` pada Python biasa |


In [6]:
arr = np.array([3, -7, 12, -4, 9, 25])

# Filter: nilai antara 0 dan 20
print(arr[(arr > 0) & (arr < 20)])     # [3, 12, 9]

# Filter: nilai negatif ATAU lebih besar dari 20
print(arr[(arr < 0) | (arr > 20)])     # [-7, -4, 25]

# Filter: nilai yang BUKAN negatif
print(arr[~(arr < 0)])                 # [3, 12, 9, 25]

[ 3 12  9]
[-7 -4 25]
[ 3 12  9 25]


## Deep Dive — Kenapa Harus Pakai &, |, ~ (Bukan and, or, not)

Ini adalah salah satu sumber error paling umum bagi orang yang baru berpindah dari Python murni ke NumPy. Penjelasannya bertingkat:

* Alasan 1 — and/or/not bawaan Python dirancang untuk nilai tunggal (skalar), bukan array. Ketika Python mengevaluasi and/or, ia akan memanggil bool() pada seluruh objek untuk menentukan True/False-nya. Pada array dengan lebih dari satu elemen, bool(arr) bersifat ambigu — apakah "benar" berarti semua elemen True, atau ada elemen yang True? NumPy sengaja menolaknya dan memunculkan error ValueError: The truth value of an array with more than one element is ambiguous.
* Alasan 2 — &, |, ~ adalah operator bitwise yang dioverload oleh NumPy agar bekerja element-wise, sama seperti operator aritmatika (+, -, dll) yang telah dipelajari di bab 1. Operator inilah yang kompatibel dengan struktur array.
* Alasan 3 — kenapa harus pakai tanda kurung () di setiap kondisi? Ini murni soal operator precedence (urutan pengerjaan operator) di Python: operator bitwise (&, |) memiliki prioritas lebih tinggi daripada operator perbandingan (>, <). Tanpa tanda kurung, ekspresi arr > 0 & arr < 20 akan dievaluasi Python sebagai arr > (0 & arr) < 20 — bukan hasil yang Anda maksud, dan hampir selalu memunculkan error atau hasil yang salah secara diam-diam (silent bug).

In [ ]:
# SALAH — tanpa tanda kurung, urutan evaluasi berantakan
arr[arr > 0 & arr < 20]     # TypeError atau hasil tidak terduga

# BENAR — setiap kondisi dibungkus tanda kurung
arr[(arr > 0) & (arr < 20)]

> Aturan praktis: setiap kali menggabungkan lebih dari satu kondisi dengan & atau |, selalu bungkus masing-masing kondisi dengan tanda kurung — tanpa terkecuali.

# Conditional Replacement

Selain untuk mengambil data, boolean array juga dapat dipakai untuk mengganti nilai secara kondisional. Ada dua pendekatan utama:



Pendekatan 1 — Boolean Assignment (mengubah array asli secara langsung, in-place):

In [2]:
import numpy as np
arr = np.array([3, -7, 12, -4, 9])
arr[arr < 0] = 0

print(arr)

[ 3  0 12  0  9]


Pendekatan 2 — np.where() (menghasilkan array baru tanpa mengubah array asli):

In [4]:
arr = np.array([3, -7, 12, -4, 9])
hasil = np.where(arr < 0,67, arr)
print(hasil)

[ 3 67 12 67  9]


### 📖 Deep Dive — np.where() Punya Dua Mode Penggunaan

np.where() sering membingungkan karena API-nya berubah tergantung jumlah argumen yang diberikan:

* Mode 1 — tiga argumen np.where(kondisi, nilai_jika_true, nilai_jika_false). Ini adalah versi vektorisasi dari ekspresi ternary (x if kondisi else y) yang bekerja element-wise. Cocok untuk transformasi kondisional tanpa loop.

In [9]:
arr = np.array([3, -7, 12, -4, 9])
arredit = np.where(arr < 0, "Negatif", "Positif")
print(arredit)

['Positif' 'Negatif' 'Positif' 'Negatif' 'Positif']


* Mode 2 — satu argumen np.where(kondisi). Mode ini tidak mengembalikan nilai array, melainkan indeks/posisi di mana kondisi bernilai True, dalam bentuk tuple.

In [10]:
arr = np.array([3, -7, 12, -4, 9])
arr_1 = np.where(arr < 0)
print(arr_1)

(array([1, 3]),)


* Perbedaan kedua mode ini penting dipahami: mode 3-argumen menjawab pertanyaan "nilai apa yang seharusnya ada di sini?", sedangkan mode 1-argumen menjawab "di posisi mana kondisi ini terpenuhi?". Kesalahan memilih mode adalah penyebab umum bug yang membingungkan bagi pemula.

# Studi Kasus: Filtering Data

In [12]:
data = np.array([12, 45, -3, 78, 200, -15, 33, 5])

# 1. Filter nilai tertentu (exact match)
print(data[data == 45])              # [45]

# 2. Filter berdasarkan threshold
print(data[data > 50])            # [78, 200]

# 3. Filter berdasarkan beberapa kondisi sekaligus
print(data[(data > 0) & (data < 50)])   # [12, 45, 33, 5]

# 4. Kombinasi filtering + agregasi (pola umum di EDA)
rata_rata = data.mean()
print(data[data > rata_rata])        # nilai di atas rata-rata

[45]
[ 78 200]
[12 45 33  5]
[ 45  78 200]


Pola data[data > data.mean()] di atas sangat umum dipakai dalam tahap eksplorasi data (EDA) untuk mengidentifikasi nilai-nilai yang berada di atas rata-rata secara cepat, tanpa perlu menulis loop manual.

# Latihan Praktik

1. Filter nilai negatif — Diberikan data = np.array([5, -3, 8, -12, 0, 7, -1]), ambil hanya elemen yang bernilai negatif.

In [15]:
data = np.array([5, -3, 8, -12, 0, 7, -1])
data_negatif = data[data < 0]
print(data_negatif)

[ -3 -12  -1]


2. Filter nilai di atas rata-rata — Gunakan array yang sama, filter elemen yang nilainya lebih besar dari data.mean().

In [17]:
avg_data = data.mean()
data_upavg = data[data > avg_data]
print(f"AVG data: {avg_data:.2f}")
print(f"Nilai di atas avg: {data_upavg}")

AVG data: 0.57
Nilai di atas avg: [5 8 7]


3. Replace nilai tertentu — Ganti seluruh nilai negatif pada array tersebut menjadi 0 menggunakan boolean assignment, lalu ulangi dengan np.where().

In [18]:
data_1 = np.where(data < 0, 0, data)
print(data_1)

[5 0 8 0 0 7 0]


4. Deteksi outlier sederhana — Gunakan aturan IQR (Interquartile Range): hitung Q1 (persentil ke-25) dan Q3 (persentil ke-75) dengan np.percentile(), lalu tentukan batas bawah (Q1 - 1.5*IQR) dan batas atas (Q3 + 1.5*IQR). Filter elemen yang berada di luar batas tersebut sebagai outlier.

In [28]:
data = np.array([50, -3, 8, -120, 0, 7, -1])
Q1 = np.percentile(data, 25)
Q3 = np.percentile(data, 75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(lower_bound)
print(upper_bound)

data_iqr = np.where((data < lower_bound) | (data > upper_bound))  # pakai | bukan &
print(data_iqr)
print(data[data_iqr])  # lihat nilai outlier-nya langsung

-16.25
21.75
(array([0, 3]),)
[  50 -120]
